In [2]:
import sys
sys.path.append('../..')
from sqlalchemy import create_engine, MetaData, select, Table
import schedule
import pandas as pd
from datetime import datetime
import time
from config import POSTGRES_UTEA
import math

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

ENGINE = create_engine(f'postgresql+psycopg://{USER_DB}:{PASS_DB}@{HOST_DB}:{PORT_DB}/{NAME_DB}')

metadata = MetaData()
reporte_tbl = Table("reporte", metadata, autoload_with=ENGINE, schema="datos_iag")
msj_whatsapp_tbl = Table("msj_whatsapp", metadata, autoload_with=ENGINE, schema="notificaciones_wsp")

In [23]:
def crear_mensaje(datos):    
    try:
        with ENGINE.begin() as conn:
            conn.execute(msj_whatsapp_tbl.insert(), datos.to_dict(orient='records'))
        print("✅ Se ha actualizado ")
    except Exception as e:
        print("❌ Error al insertar en tabla MSJ WHASTAPP", e)
    #return df

In [24]:
df = pd.read_excel(r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Trichogramma\2026\ANALISIS DE AREA PARA RECOMENDACION\RESUMEN_LOTES_RECOMENDACION.xlsx', sheet_name = 'PRIMEROS_58')

In [25]:
df

,cod_canero,nombre_canero,numero_cel,link
0,1,AGUILERA OLGA RIVERO VDA DE,77000503,https://drive.google.com/file/d/1JrK-_-j5n8J-h...
1,1,AGUILERA OLGA RIVERO VDA DE,75067427,https://drive.google.com/file/d/1JrK-_-j5n8J-h...
2,5,AGROPECUARIA MARIANA S.R.L.,76607676,https://drive.google.com/file/d/1sBGLvUoNPp0b4...
3,6,FRERKING ORTIZ RICARDO,77049491,https://drive.google.com/file/d/1C7WSeff8PrNUL...
4,7,NAZER PARADA ROBERTO DOMINGO,77624793,https://drive.google.com/file/d/1lUKwwgJ1Lkc1w...
...,...,...,...,...
60,433,TREPP CARRASCO RUDIGER,76615363,https://drive.google.com/file/d/1jybrXTW9p4vQ7...
61,452,CORDOVA JUSTINIANO GERSON MOISES,78061560,https://drive.google.com/file/d/1ZtGyEZ6Mk60Ja...
62,501,PALICIO PEREZ CARREÑO ADRIANA NICOLE,78423141,https://drive.google.com/file/d/1qtUjO4wWY16nr...
63,529,ALVAREZ SEJAS BISMARCK,67188780,https://drive.google.com/file/d/1f_wKc77mtVPE-...


In [26]:
df['fecha_registro'] = datetime.now()

df['mensaje'] = df.apply(lambda row: f"""🌱 *Programa de Control Biológico En Caña de Azucal* 🚜

Saludos estimado productor cañero(a): *{row['nombre_canero']}* 👋, le escribimos desde UTEA - GUABIRA para hacerle llegar el siguiente informe con respecto al Programa de Control Biologico 2026-2027. 📄📍

{row['link']}

Si tiene alguna duda o consulta puede comunicarse con Ing. José Armando Casanova al número de teléfono/WhatsApp: 68908131. 📲
Quedamos atentos a sus consultas. 🙏✨

> Si no puede abrir el enlace del informe, pruebe agregar este número a sus contactos.!""", axis=1)

df['enviado'] = False
df['fecha_enviado'] = None
df['numero_contac'] = '591' + df['numero_cel'].astype(str) + '@s.whatsapp.net'

In [24]:
df = df.drop(['COD_COS', 'NOMBRE', 'NUM', 'OBS'], axis=1)

In [32]:
df.head(2)

,cod_canero,nombre_canero,numero_cel,link,fecha_registro,mensaje,enviado,fecha_enviado,numero_contac
0,1,AGUILERA OLGA RIVERO VDA DE,77000503,https://drive.google.com/file/d/1JrK-_-j5n8J-h...,2026-04-05 23:21:38.775310,🌱 *Programa de Control Biológico En Caña de Az...,False,None,59177000503@s.whatsapp.net
1,1,AGUILERA OLGA RIVERO VDA DE,75067427,https://drive.google.com/file/d/1JrK-_-j5n8J-h...,2026-04-05 23:21:38.775310,🌱 *Programa de Control Biológico En Caña de Az...,False,None,59175067427@s.whatsapp.net


In [19]:
#crear_mensaje(df)

In [35]:
len(df)

65

In [36]:
contador = 0
for index, row in df.iterrows():
    df_fila = pd.DataFrame([row])
    df_fila['fecha_registro'] = datetime.now()
    crear_mensaje(df_fila)
    contador = contador + 1
    print('Contador:', contador, ' - ' + str(row['cod_canero']) + ' / ' + row['nombre_canero'])
    time.sleep(60)

✅ Se ha actualizado 
Contador: 1  - 1 / AGUILERA OLGA RIVERO VDA DE
✅ Se ha actualizado 
Contador: 2  - 1 / AGUILERA OLGA RIVERO VDA DE
✅ Se ha actualizado 
Contador: 3  - 5 / AGROPECUARIA MARIANA S.R.L.


KeyboardInterrupt: 